# Tier 2 — the Natural branch: where will lightning fire burn?

Tier 1 ([`06_analysis.ipynb`](06_analysis.ipynb)) established that **Natural (lightning) is 59% of
all burned area** — and it is a cause the planner cannot prevent. The only lever is **mitigation**
(fuel treatment, defensible space, suppression pre-positioning), which is about **where** the burn
lands, not what ignited it. So unlike the Human branch this is not a cause-composition problem —
Natural is effectively one cause — it is a **location** problem: which region-seasons will carry the
Natural burn next season?

**Deliverable.** A forward-looking map of expected Natural burned area across region-seasons, at
EPA Level III grain. Whether a finer grain is needed is left open.

**Input.** `data/region_season_cause.parquet`, plus `data/region_season_climate.parquet` for the
covariate section.

Same grain, partial-winter boundary rule, and forward-chaining split as the rest of the pipeline.
Per the design, this branch is methodologically distinct from the RQ2 forecast (Tier 1 + the Human
branch); it tells the planner what the cause-risk profile *cannot* act on by prevention.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, "../src")
from config import ProjectConfig
from panel import RegionSeasonPanel

# Paths, the boundary rule and the forward-chaining split come from src/config.py.
cfg = ProjectConfig()
DATA = cfg.data

# `natural_acres()` returns Natural burned acres per region-season, including zero cells.
# Filtering to nat_ac > 0 is this branch's own decision: that positive subset is a distinct
# cell population and the forward-chaining masks below are computed on it.
panel = RegionSeasonPanel.load(cfg)
nat = panel.natural_acres()

print(f"{len(nat):,} region-season cells | {(nat.nat_ac > 0).sum():,} with Natural burn > 0")
print(f"per-cell Natural acres: median {nat.loc[nat.nat_ac>0,'nat_ac'].median():,.0f}, "
      f"max {nat.nat_ac.max():,.0f}  (heavy-tailed -> log-space scoring)")

10,135 region-season cells | 6,662 with Natural burn > 0
per-cell Natural acres: median 41, max 2,826,326  (heavy-tailed -> log-space scoring)


## How concentrated is Natural burn across regions?

**Hypothesis.** Targeted mitigation is only worthwhile if Natural burn concentrates spatially. If it
were spread uniformly, siting would have nothing to target.

**Experiment.** For each season-year, two concentration measures across regions: the **Herfindahl
index (HHI)** of Natural acres (1 = one region has it all, ~0 = spread evenly) and the **top-region
share**.

In [2]:
def hhi(x):
    s = x.sum()
    return np.nan if s <= 0 else float(((x / s) ** 2).sum())

def top_share(x):
    s = x.sum()
    return np.nan if s <= 0 else float(x.max() / s)

pos = nat[nat["nat_ac"] > 0]
conc = pos.groupby(["season", "season_year"])["nat_ac"].agg(HHI=hhi, top_share=top_share)

by_season = conc.groupby("season").mean()
by_season["n_regions_burning"] = pos.groupby(["season", "season_year"]).size().groupby("season").mean()
print("Natural-burn concentration across regions, mean over years, by season:\n")
print(by_season.round(3).to_string())
print("\nSummer (JJA) spreads across many western regions (low HHI); winter/spring concentrate in a")
print("few (high HHI, top region ~half) -- so where to mitigate is itself season-dependent.")

Natural-burn concentration across regions, mean over years, by season:

          HHI  top_share  n_regions_burning
season                                     
DJF     0.380      0.510             23.679
JJA     0.137      0.262             81.655
MAM     0.345      0.497             64.793
SON     0.248      0.392             60.414

Summer (JJA) spreads across many western regions (low HHI); winter/spring concentrate in a
few (high HHI, top region ~half) -- so where to mitigate is itself season-dependent.


## Can next-season Natural acres per region be predicted?

**Hypothesis.** If a region's Natural burn is predictable from its own history, the biggest expected
burdens can be ranked for mitigation ahead of the season.

**Experiment.** Predict `log10(nat_ac)` with a forward-chaining trailing mean, scored on the
held-out tail (season-year ≥ 2010) with log-MAE, reported unweighted and **acre-weighted**
("typically off by a factor of ~N"). Compared against a deliberately uninformed **global prior** —
one constant for every cell, fit on training years only.

Cells with zero Natural burn are excluded from the log target (many non-western cells never see
lightning fire); the question is *how much*, given that a region burns at all.

In [3]:
TEST_START = cfg.test_start     # forward-chaining split, shared across all branches
K = cfg.shares_k                # trailing window locked by the Tier-1 shares sweep

# TrailingMean does shift(1) then a k-window mean within (region, season), asserting the frame
# is sorted first. GlobalPrior is the uninformed reference: one constant everywhere, fit on
# training years only.
from trailing import GlobalPrior, TrailingMean

natp = nat[nat["nat_ac"] > 0].copy()
natp["log_nat"] = np.log10(natp["nat_ac"])
natp = natp.sort_values(list(cfg.sort_keys)).reset_index(drop=True)

actual = natp["log_nat"].to_numpy()
w = natp["nat_ac"].to_numpy()                           # acre weight (rank the big burdens)
in_test = (natp["season_year"] >= TEST_START).to_numpy()
train = (natp["season_year"] < TEST_START).to_numpy()

def logmae(pred):
    err = np.abs(pred - actual)
    m = in_test & ~np.isnan(err)
    return {"n_cells": int(m.sum()),
            "logMAE_unwtd": float(err[m].mean()),
            "logMAE_acre_wtd": float(np.average(err[m], weights=w[m])),
            "x_off_acre_wtd": float(10 ** np.average(err[m], weights=w[m]))}

pred_trail = TrailingMean(K).predict(natp, "log_nat")["log_nat"].to_numpy()

# Reference: a global 'typical Natural acres' prior from training years only (no region
# info). Acre-weighted, so it reflects where the acres actually are.
prior = GlobalPrior(weighted=True).fit(natp, "log_nat", train_mask=train, weight_col="nat_ac")
global_log = float(prior.value_[0])
pred_global = prior.predict(natp)["log_nat"].to_numpy()

res = pd.DataFrame([logmae(pred_global), logmae(pred_trail)],
                   index=["global prior (train mean)", f"persistence (k={K})"])
print(f"NATURAL location -- log-space, held-out tail season_year >= {TEST_START}\n")
print(res.round(4).to_string())

print("\nRead this split carefully -- the two metrics disagree, and the disagreement is the finding:")
print("  * UNWEIGHTED: persistence wins big (it nails the many small/typical cells).")
print("  * ACRE-WEIGHTED: the global prior wins -- because on the megafire cells that carry the")
print("    acres, a region's own calm-year history badly UNDER-predicts its record year, while a")
print("    high global constant lands closer. History is actively misleading on the big burns.")

NATURAL location -- log-space, held-out tail season_year >= 2010

                           n_cells  logMAE_unwtd  logMAE_acre_wtd  x_off_acre_wtd
global prior (train mean)     2727        3.7638           0.4998          3.1609
persistence (k=7)             2708        0.9141           1.1463         14.0044

Read this split carefully -- the two metrics disagree, and the disagreement is the finding:
  * UNWEIGHTED: persistence wins big (it nails the many small/typical cells).
  * ACRE-WEIGHTED: the global prior wins -- because on the megafire cells that carry the
    acres, a region's own calm-year history badly UNDER-predicts its record year, while a
    high global constant lands closer. History is actively misleading on the big burns.


In [4]:
# The biggest held-out Natural cells, and how each predictor does on them.
m = in_test & ~np.isnan(pred_trail)
diag = natp[m].copy()
diag["persist_log"] = pred_trail[m]
diag["global_log"] = global_log
diag["err_persist"] = np.abs(diag["persist_log"] - diag["log_nat"])
diag["err_global"] = np.abs(diag["global_log"] - diag["log_nat"])

top = diag.nlargest(6, "nat_ac")[
    ["region", "season", "season_year", "nat_ac", "log_nat",
     "persist_log", "global_log", "err_persist", "err_global"]]
print(f"global prior = {global_log:.2f} log-acres (~{10**global_log:,.0f} ac); "
      f"test-cell median = {np.median(actual[m]):.2f} log-acres (~{10**np.median(actual[m]):,.0f} ac)\n")
print("The six largest held-out Natural burns -- persistence under-predicts every one by 1-1.7 orders:")
print(top.round(2).to_string(index=False))
print("\nThis is the Tier-1 'can't see a megafire coming' result in its sharpest form, and the case")
print("for external pre-season covariates: the acres live in years history cannot anticipate.")

global prior = 5.46 log-acres (~285,412 ac); test-cell median = 1.55 log-acres (~35 ac)

The six largest held-out Natural burns -- persistence under-predicts every one by 1-1.7 orders:
                                             region season  season_year     nat_ac  log_nat  persist_log  global_log  err_persist  err_global
             Interior Forested Lowlands and Uplands    JJA         2015 2826326.39     6.45         4.73        5.46         1.72        1.00
                           Northern Basin and Range    JJA         2012 2115903.79     6.33         5.17        5.46         1.15        0.87
                               Interior Bottomlands    JJA         2015 1278711.00     6.11         4.60        5.46         1.50        0.65
Klamath Mountains/California High North Coast Range    JJA         2020 1176272.43     6.07         4.68        5.46         1.39        0.62
             Interior Forested Lowlands and Uplands    JJA         2019 1161385.00     6.06         5.30 

## Does escape propensity separate regions better than magnitude?

**Hypothesis.** A region's Natural acres in a season are roughly *how many lightning fires started*
times *how big they got*. Persistence attacks that product directly. Decomposing it asks a different
question: **which regions cannot stop a fire once it starts?**

That is a property of fuel continuity, terrain, access and suppression reach — and it is the
property mitigation acts on. Fuel treatment does not reduce lightning strikes or change the weather;
it changes whether an ordinary fire stays ordinary. If escape propensity is more stable across the
forward-chaining split than raw magnitude, it is the better ranking signal.

**Experiment.** Per region, pooled over training years only: total Natural starts, total Natural
acres, and acres per start. Then a stability test — does a region's training-years ratio hold in the
held-out years — scored against the same test applied to raw magnitude, so the comparison is
like-for-like.

**Two limits, stated up front.** FPA-FOD records one row per fire at its *final* size, so this
observes the endpoint distribution, not a fire becoming large. And acres-per-start confounds fire
behavior with suppression effort: a region may burn big because fuels are continuous *or* because it
is remote and unstaffed. Those are not separable in this dataset.

In [5]:
# natural_starts() adds nat_fires alongside nat_ac; natural_acres() is untouched, so `actual`,
# `w` and `in_test` above still hold.
nstart = panel.natural_starts()
nstart_tr = nstart[nstart["season_year"] < TEST_START]      # train years only -- no peeking

# Pooled over all training season-years: the claim is that escape is a STRUCTURAL property,
# not a per-season one.
esc = (nstart_tr.groupby("region", observed=True)[["nat_ac", "nat_fires"]].sum()
       .rename(columns={"nat_ac": "acres_tr", "nat_fires": "starts_tr"}))
esc = esc[esc["starts_tr"] >= 30]                            # a ratio needs a denominator
esc["ac_per_start"] = esc["acres_tr"] / esc["starts_tr"]

print(f"{len(esc)} regions with >= 30 Natural starts in train years "
      f"({nstart_tr.season_year.min()}-{nstart_tr.season_year.max()})\n")
print("acres per Natural start, across regions:")
print(esc["ac_per_start"].describe(percentiles=[.1, .25, .5, .75, .9]).round(2).to_string())
print(f"\nspread: {esc['ac_per_start'].max() / esc['ac_per_start'].min():,.0f}x "
      "between the highest and lowest region")

cols = ["starts_tr", "acres_tr", "ac_per_start"]
print("\nHIGHEST escape propensity -- fires that start here run:")
print(esc.nlargest(8, "ac_per_start")[cols].round(1).to_string())
print("\nLOWEST -- fires that start here get caught:")
print(esc.nsmallest(8, "ac_per_start")[cols].round(1).to_string())

87 regions with >= 30 Natural starts in train years (1992-2009)

acres per Natural start, across regions:
count       87.00
mean       968.78
std       2708.52
min          0.59
10%          1.95
25%         22.07
50%         68.85
75%        418.32
90%       2005.35
max      14828.75

spread: 25,108x between the highest and lowest region

HIGHEST escape propensity -- fires that start here run:
                                        starts_tr   acres_tr  ac_per_start
region                                                                    
Yukon Flats                                   178  2639517.3       14828.7
Ogilvie Mountains                              68   885577.2       13023.2
Interior Highlands                            467  4426852.8        9479.3
Interior Bottomlands                          434  3961901.3        9128.8
Interior Forested Lowlands and Uplands       1062  9419942.3        8870.0
Arctic Foothills                               75   503129.4        6708.4
Su

In [6]:
# Does a region's train-years ratio predict its held-out ratio?
from scipy.stats import spearmanr

nstart_te = nstart[nstart["season_year"] >= TEST_START]
esc_te = (nstart_te.groupby("region", observed=True)[["nat_ac", "nat_fires"]].sum()
          .rename(columns={"nat_ac": "acres_te", "nat_fires": "starts_te"}))
esc_te = esc_te[esc_te["starts_te"] >= 30]
esc_te["ac_per_start_te"] = esc_te["acres_te"] / esc_te["starts_te"]

both = esc[["ac_per_start"]].join(esc_te[["ac_per_start_te"]], how="inner")
rho_esc = spearmanr(both["ac_per_start"], both["ac_per_start_te"]).statistic

# The like-for-like comparison: how well a region's train-years total Natural ACREAGE predicts
# its held-out acreage, measured the same way on the same regions.
mag = (nstart_tr.groupby("region", observed=True)["nat_ac"].sum().rename("ac_tr")
       .to_frame().join(
           nstart_te.groupby("region", observed=True)["nat_ac"].sum().rename("ac_te"),
           how="inner").loc[both.index])
rho_mag = spearmanr(mag["ac_tr"], mag["ac_te"]).statistic

print(f"{len(both)} regions with >= 30 Natural starts in BOTH train and held-out years\n")
print(f"  escape propensity (acres/start)  train -> held-out Spearman rho = {rho_esc:.3f}")
print(f"  raw magnitude     (total acres)  train -> held-out Spearman rho = {rho_mag:.3f}")
print("\nBoth are stable, and magnitude is if anything the MORE stable of the two -- so escape")
print("propensity is not a better ranking signal than acreage at the region level. Pooling over")
print("years averages out the bad-year volatility the ablation ladder hit: that volatility is")
print("per-season-year, not per-region. What the decomposition buys is interpretation, not lift --")
print("it separates regions that burn because they have many starts from regions that burn because")
print("their fires run, and those point at different mitigation responses.")

# Which of the two drives a region's rank: split the top-acreage regions on their source.
top = esc.nlargest(12, "acres_tr").copy()
top["starts_pctile"] = esc["starts_tr"].rank(pct=True).loc[top.index]
top["escape_pctile"] = esc["ac_per_start"].rank(pct=True).loc[top.index]
print("\nThe 12 biggest-burning regions, decomposed (percentile within the 87 regions):")
print(top[["acres_tr", "starts_tr", "ac_per_start", "starts_pctile", "escape_pctile"]]
      .round(2).to_string())

83 regions with >= 30 Natural starts in BOTH train and held-out years

  escape propensity (acres/start)  train -> held-out Spearman rho = 0.909
  raw magnitude     (total acres)  train -> held-out Spearman rho = 0.932

Both are stable, and magnitude is if anything the MORE stable of the two -- so escape
propensity is not a better ranking signal than acreage at the region level. Pooling over
years averages out the bad-year volatility the ablation ladder hit: that volatility is
per-season-year, not per-region. What the decomposition buys is interpretation, not lift --
it separates regions that burn because they have many starts from regions that burn because
their fires run, and those point at different mitigation responses.

The 12 biggest-burning regions, decomposed (percentile within the 87 regions):
                                          acres_tr  starts_tr  ac_per_start  starts_pctile  escape_pctile
region                                                                          

**Finding.** Escape propensity is real and stable, but it is **not** a better ranking signal than
acreage.

Acres per Natural start spans **25,108×** across the 87 regions with enough starts to form a ratio,
from Yukon Flats at 14,829 ac/start down to Puget Lowland at 0.59. It is highly stable across the
split: Spearman **ρ = 0.909** between a region's training-years ratio and its held-out ratio.

But the same test on raw magnitude gives **ρ = 0.932**, slightly *higher*. Both quantities are
stable at region level, so decomposing magnitude into starts × escape does not recover ranking skill
that magnitude alone lacked.

**This does not contradict the persistence result.** That failed on per-**region-season-year**
magnitude; this is per-**region** magnitude pooled across many years, which averages out the bad-year
volatility. Both are true: *which* regions carry the Natural burn is highly predictable, *when* they
carry it is not.

**What the decomposition does buy** is a distinction between two structurally different routes to the
top of the burned-area ranking, which imply different mitigation responses:

| Region | acres (train) | starts | ac/start | starts pctile | escape pctile |
| --- | ---: | ---: | ---: | ---: | ---: |
| Interior Forested Lowlands and Uplands | 9.4M | 1,062 | 8,870 | 0.64 | 0.95 |
| Central Basin and Range | 6.0M | 12,073 | 500 | 0.95 | 0.79 |
| Interior Highlands | 4.4M | 467 | 9,479 | 0.46 | 0.98 |
| Yukon Flats | 2.6M | 178 | 14,829 | 0.30 | 1.00 |
| Arizona/New Mexico Mountains | 2.2M | 25,452 | 86 | 1.00 | 0.53 |
| Blue Mountains | 1.5M | 11,611 | 133 | 0.94 | 0.61 |

**Yukon Flats** sits at the 30th percentile for ignition count and the 100th for escape — 2.6M acres
from 178 starts. **Arizona/New Mexico Mountains** is the mirror image: the most lightning starts of
any region (25,452) but median escape (86 ac/start), reaching comparable acreage by volume of
ignitions rather than by fires running. A planner reading only the acreage ranking cannot tell that
one needs fuel continuity and access work while the other is already catching most of what starts.

**The confound is concentrated, not diffuse.** The top of the escape ranking is dominated by Alaskan
interior ecoregions (Yukon Flats, Ogilvie Mountains, Interior Highlands, Interior Bottomlands,
Interior Forested Lowlands), where "fuels are continuous" and "nobody is there to suppress it" are
both true and inseparable in FPA-FOD. Read the measure as *observed* escape — what happens to a fire
that starts here — not as a fuels property.

Ranking regions by expected Natural acreage is defensible (ρ = 0.93 across the split), and the
starts-vs-escape split is the natural annotation on that ranking.

## Do external drought and fuel-dryness covariates beat the persistence floor?

**Hypothesis.** A big lightning-fire year is driven by *that season's* dryness and fuel state, which
history cannot see. Persistence under-predicts **every one** of the six largest held-out Natural
cells by 1–1.7 orders of magnitude, so pre-season drought data should carry information history
lacks.

**Source: TerraClimate** (Climatology Lab, U. Idaho), 1/24° (~4 km) monthly grids, 1958–present,
pulled via THREDDS/OPeNDAP. Loader: [`../src/terraclimate.py`](../src/terraclimate.py); artifact:
`data/region_season_climate.parquet`.

**Why this source.** The analysis grain includes **20 Alaska** Level III ecoregions, which eliminated
both standard drought products: **gridMET PDSI** and **nClimGrid SPEI/PDSI** are CONUS-only and would
have silently dropped every AK cell. TerraClimate is global terrestrial.

**Why fuel *condition*, not fuel *load*.** LANDFIRE was rejected for this panel: its base map is circa
2001 with discrete vintages and usable Alaska coverage only from the 2016 Remap, so against a
1992–2020 season-year panel it contributes almost no *interannual* variance — precisely the variance
the megafire-year problem needs explained.

| Covariate | TerraClimate var | What it carries |
| --- | --- | --- |
| `pdsi` | `PDSI` | Palmer Drought Severity Index |
| `soil_moisture` | `soil` | column soil moisture (mm) — antecedent wetness |
| `water_deficit` | `def` | climatic water deficit (mm) — dryness with an energy term |
| `vpd` | `vpd` | vapor pressure deficit (kPa) — atmospheric dryness |

**Leakage rule.** Each cell value is the mean over the **3 calendar months immediately preceding the
target season's first month**. Nothing from within the target season is read. The winter case is the
trap: DJF of year *Y* begins in **December of *Y−1***, so its pre-season window is Sep–Nov of *Y−1*,
not of *Y*. Getting that wrong would leak a full year of future climate into every winter cell. The
rule is asserted in `terraclimate._self_check()` and re-verified below.

Spatial aggregation is an **area-weighted** mean over grid centers inside each ecoregion polygon,
weighted by cos(latitude) — material at Alaska's latitudes.

In [7]:
import sys
sys.path.insert(0, str(Path("..") / "src"))
import terraclimate as tc

# Re-assert the leakage rule here rather than trusting the module: an off-by-one on the DJF
# year boundary would leak 12 months of future climate into every winter cell.
tc._self_check()

clim = pd.read_parquet(DATA / "region_season_climate.parquet")
COVS = ["pdsi", "soil_moisture", "water_deficit", "vpd"]

print(f"\nclimate table: {clim.shape[0]:,} region-season cells x {len(COVS)} covariates")
print(f"regions {clim['region'].nunique()} | season_year {clim['season_year'].min()}-{clim['season_year'].max()}")
print(f"missing per covariate:\n{clim[COVS].isna().sum().to_string()}")

# Left join: every natp cell keeps its row, and an unresolved covariate shows up as NaN
# rather than silently dropping a fire cell.
natx = natp.merge(clim[["region", "season", "season_year"] + COVS],
                  on=["region", "season", "season_year"], how="left")
assert len(natx) == len(natp), "join changed row count -- key collision"

matched = natx[COVS].notna().all(axis=1)
print(f"\njoined: {matched.sum():,}/{len(natx):,} cells have all four covariates "
      f"({matched.mean():.1%})")

terraclimate self-check passed (leakage rule + season spine)

climate table: 12,075 region-season cells x 4 covariates
regions 105 | season_year 1992-2020
missing per covariate:
pdsi             0
soil_moisture    0
water_deficit    0
vpd              0

joined: 6,662/6,662 cells have all four covariates (100.0%)


### The ablation ladder

The covariates enter on **exactly the same terms** as the baselines: same cells, same
forward-chaining split (`season_year >= 2010` held out), same log-MAE reported unweighted and
acre-weighted. Any movement is attributable to the external data alone.

Two rungs, in ablation order:

1. **climate only** — the four covariates, no fire history. Tests whether pre-season dryness alone
   carries the signal.
2. **persistence + climate** — the trailing log-acres feature plus the covariates. Tests whether the
   external data adds anything *on top of* what history already knew.

A gradient-boosted tree rather than linear, because the expected relationship is threshold-like:
burned area does not respond linearly to PDSI, it responds once a region crosses into drought.

The floor failed specifically on the **acre-weighted** metric — the megafire cells — so the question
is whether acre-weighted error comes down, not whether average log-MAE improves.

In [8]:
from sklearn.ensemble import HistGradientBoostingRegressor

# The scoring frame is natx (natp + covariates), so `actual`, `w` and `in_test` from the
# baseline cell still align row-for-row.
assert (natx["log_nat"].to_numpy() == actual).all(), "row alignment broke"

trail_feat = pred_trail                       # the persistence prediction, reused as a feature
X_clim = natx[COVS].to_numpy()
X_both = np.column_stack([trail_feat, X_clim])

def fit_score(X, label):
    """Fit on training years, score the held-out tail on the same metric as the floor."""
    ok = ~np.isnan(X).any(axis=1)
    tr = train & ok
    model = HistGradientBoostingRegressor(
        max_depth=3, max_iter=300, learning_rate=0.05, random_state=0)
    model.fit(X[tr], actual[tr])
    pred = np.full(len(actual), np.nan)
    pred[ok] = model.predict(X[ok])
    return logmae(pred) | {"model": label}

rows = [
    logmae(pred_global) | {"model": "global prior (train mean)"},
    logmae(pred_trail) | {"model": f"persistence (k={K})"},
    fit_score(X_clim, "climate only (4 covariates)"),
    fit_score(X_both, "persistence + climate"),
]
ladder = pd.DataFrame(rows).set_index("model")[
    ["n_cells", "logMAE_unwtd", "logMAE_acre_wtd", "x_off_acre_wtd"]]

print(f"ABLATION LADDER -- Natural acres, held-out season_year >= {TEST_START}\n")
print(ladder.round(4).to_string())

base = ladder.loc[f"persistence (k={K})", "x_off_acre_wtd"]
best = ladder["x_off_acre_wtd"].min()
print(f"\nacre-weighted: persistence floor {base:.1f}x off -> best rung {best:.1f}x off")
print("The metric that matters is x_off_acre_wtd -- that is where the floor failed.")

ABLATION LADDER -- Natural acres, held-out season_year >= 2010

                             n_cells  logMAE_unwtd  logMAE_acre_wtd  x_off_acre_wtd
model                                                                              
global prior (train mean)       2727        3.7638           0.4998          3.1609
persistence (k=7)               2708        0.9141           1.1463         14.0044
climate only (4 covariates)     2727        1.3345           2.8873        771.4717
persistence + climate           2708        0.9243           1.2273         16.8774

acre-weighted: persistence floor 14.0x off -> best rung 3.2x off
The metric that matters is x_off_acre_wtd -- that is where the floor failed.


**Finding.** Adding the four covariates to the persistence feature left held-out error essentially
unchanged — logMAE 0.914 → 0.924 unweighted, 14.0× → 16.9× acre-weighted. The model had the drought
and dryness signals available and leaned on the trailing fire-history feature anyway.

Given a region's own recent Natural-acres history, pre-season dryness **at this grain, pooled across
all regions**, contributed no additional predictive information.

**Scope of the claim.** What holds is narrow: *three-month pre-season drought and fuel-dryness means,
aggregated area-weighted to EPA Level III, pooled in one model, add nothing to what a region's own
7-year trailing history already implies about its next-season Natural acres.* That is not "drought
does not drive lightning-fire area." Several explanations are consistent with it — history and
drought may carry the same slow regional signal; ecoregion-scale averaging may dilute a signal real
at finer grain; a single pre-season window cannot express a wet-year-then-dry-spring curing
mechanism; and the pooled fit may average away region-varying relationships.

**That last explanation is tested below and confirmed.** Disaggregated per region, the relationship
ranges from |ρ| = 0.09 to 0.53 and *inverts sign* in arid regions, so this pooled model was asked to
hold opposite mechanisms in one function. The negative result stands **for the pooled deliverable**;
it should not be cited as "pre-season dryness is uninformative."

**Two scoring caveats.**

1. **The `climate only` rung's 771× is a metric artifact.** That model trained on years where the
   typical cell is small (held-out median ≈ 35 acres), so it learned to predict small; the
   acre-weighted metric then grades it with weights proportional to `nat_ac`, where a handful of
   megafire cells dominate. Its unweighted logMAE of 1.33 is mediocre, not broken. The informative
   rung for the drought question is `persistence + climate`.

2. **The `global prior` rung is fit acre-weighted and scored acre-weighted**, so its apparent win is
   partly circular. It lands at 5.46 log-acres (~285,000 ac) against a held-out median of 1.55
   (~35 ac) — a constant tuned to the large cells, then graded with the large cells weighted most.
   Its unweighted logMAE of 3.76 shows the cost elsewhere.

**What stands independently.** Persistence under-predicts each of the six largest held-out Natural
burns by 1–1.7 orders of magnitude. Whether a lightning strike coincides with a wind event is not
recoverable from any pre-season covariate, and next-season Natural *magnitude* at region-season grain
is not predictable from the fire record, with or without drought.

The deliverable is mitigation siting — *where* the burn will concentrate — which is a ranking
question, while log-MAE scores magnitude. A model can rank the right regions and still post a poor
MAE. The concentration result (JJA HHI 0.137 across ~82 burning regions vs DJF 0.380 with the top
region carrying ~half) remains the strongest evidence here and is descriptive only.

The persistence rungs score 2,708 held-out cells against 2,727 for the others, since the trailing
window cannot predict the first *k* cells of each (region, season) group. This does not drive the
comparison but should be equalized before any rung is reported as a headline number.

## Was the drought failure uniform across regions?

**Hypothesis.** The ablation pooled **all 105 regions into one model** and reported one number. That
cannot distinguish two worlds:

1. Pre-season dryness is uninformative **everywhere**.
2. Pre-season dryness is informative in **some** regions, inverted in others, and pooling averages
   the two to nothing.

These imply opposite next steps. Under (1) the external-covariate direction is closed. Under (2) the
covariates are real but the *grain of the model* is wrong, and the fix is stratification, not better
data.

**Experiment.** Restrict to **JJA**, which carries the overwhelming majority of Natural acres, and to
cells with `nat_ac > 0`. For each of the 20 largest-burning regions independently, compute the
Spearman correlation between each pre-season covariate and `log10(nat_ac)` across that region's 29
season-years. Spearman rather than Pearson because the relationship is expected to be monotone but
not linear, and rank correlation is robust to the heavy tail.

**Sign expectations**, so coherence can be distinguished from noise: `water_deficit` and `vpd`
measure dryness, so both should be **positive** (drier → more burn). `pdsi` and `soil_moisture`
measure wetness, so both should be **negative**.

This is a **descriptive diagnostic on the full record, not a predictive test** — all 29 years, no
forward-chaining split — so it cannot be read as held-out skill. It answers "is there a within-region
relationship at all," the prior question to "can it be used."

The 20 regions are chosen by total JJA Natural acres, so this is a conditional look at the regions
that matter, not an unbiased survey of all 105. And 20 regions × 4 covariates is 80 correlations, so
individual large values are expected by chance; what carries weight is whether the *pattern* within a
region is coherent.

In [9]:
# Which regions carry the JJA Natural acres, and how volatile each is year to year. Total acres
# alone picks the biggest region but says nothing about whether there is variance to explain.
jja = nat[nat["season"] == "JJA"]
g = jja.groupby("region", observed=True)["nat_ac"]

vol = pd.DataFrame({
    "total_ac": g.sum(),
    "mean_ac": g.mean(),
    "max_ac": g.max(),
    "n_yr_pos": jja[jja["nat_ac"] > 0].groupby("region", observed=True).size(),
})
vol["cv"] = g.std() / g.mean()                    # volatility on the raw acre scale

# log_sd matters most: the branch models log10(nat_ac), so spread in log space is the variance
# a covariate would be asked to explain. Positive cells only, matching the modeling frame.
lg = jja[jja["nat_ac"] > 0].copy()
lg["log_nat"] = np.log10(lg["nat_ac"])
vol["log_sd"] = lg.groupby("region", observed=True)["log_nat"].std()

# How much of a region's record sits in its single worst year: the same concentration helpers
# applied across years within a region instead of across regions within a year.
vol["yr_hhi"] = g.apply(hhi)
vol["top_yr_share"] = g.apply(top_share)

TOPN = 20
top_vol = vol.nlargest(TOPN, "total_ac")

print(f"TOP {TOPN} ECOREGIONS BY TOTAL JJA NATURAL ACRES (full record, 1992-2020)\n")
print(top_vol[["total_ac", "mean_ac", "max_ac", "cv", "log_sd", "top_yr_share", "n_yr_pos"]]
      .round(2).to_string())
print("\nlog_sd is the column to read: it is the interannual spread on the scale this branch")
print("models. Total acres and log_sd are NOT the same ranking -- the biggest burner is not")
print("the most volatile, and volatility is what a pre-season covariate would have to explain.")

TOP 20 ECOREGIONS BY TOTAL JJA NATURAL ACRES (full record, 1992-2020)

                                                        total_ac    mean_ac      max_ac    cv  log_sd  top_yr_share  n_yr_pos
region                                                                                                                       
Interior Forested Lowlands and Uplands               14504881.39  500168.32  2826326.39  1.56    0.93          0.19      29.0
Northern Basin and Range                              8957487.07  308878.86  2115903.79  1.40    0.72          0.24      29.0
Central Basin and Range                               7524663.10  259471.14  1326096.35  1.29    0.64          0.18      29.0
Idaho Batholith                                       6000492.31  206913.53  1213891.82  1.51    0.82          0.20      29.0
Interior Bottomlands                                  5761289.40  198665.15  1278711.00  1.62    0.84          0.22      29.0
Interior Highlands                             

In [10]:
# Per-region Spearman between each pre-season covariate and log10 Natural acres, across that
# region's JJA season-years. Reuses `natx`, already joined under the verified leakage rule.
jjax = natx[natx["season"] == "JJA"]

MIN_YEARS = 10                                   # a 29-year panel; anything less is not a correlation
rows = []
for region in top_vol.index:
    sub = jjax[jjax["region"] == region]
    if len(sub) < MIN_YEARS:
        continue
    row = {"region": region, "n": len(sub)}
    for c in COVS:
        ok = sub[c].notna()
        row[c] = spearmanr(sub.loc[ok, c], sub.loc[ok, "log_nat"]).statistic
    rows.append(row)

percov = pd.DataFrame(rows).set_index("region")

# Coherence: does a region's sign pattern match the physical expectation?
EXPECTED = {"pdsi": -1, "soil_moisture": -1, "water_deficit": +1, "vpd": +1}
percov["n_signs_ok"] = sum((np.sign(percov[c]) == s).astype(int) for c, s in EXPECTED.items())
percov["best_abs"] = percov[COVS].abs().max(axis=1)

sign_note = ", ".join(f"{c} {'+' if s > 0 else '-'}" for c, s in EXPECTED.items())
print("WITHIN-REGION Spearman rho: pre-season covariate vs log10(JJA Natural acres)")
print(f"expected signs -- {sign_note}")
print("(descriptive, full record -- NOT the forward-chaining split)\n")
print(percov.sort_values("best_abs", ascending=False)
      [["n", *COVS, "n_signs_ok", "best_abs"]].round(3).to_string())

print("\n|rho| distribution across these regions, per covariate:")
print(percov[COVS].abs().describe().loc[["mean", "50%", "max"]].round(3).to_string())

pooled = percov["best_abs"]
print(f"\nspread in best-covariate |rho| across regions: {pooled.min():.3f} to {pooled.max():.3f}")
print("If the pooled ladder's negative result were the whole story, this column would be flat")
print("and small everywhere. It is not -- which is the finding.")

WITHIN-REGION Spearman rho: pre-season covariate vs log10(JJA Natural acres)
expected signs -- pdsi -, soil_moisture -, water_deficit +, vpd +
(descriptive, full record -- NOT the forward-chaining split)

                                                      n   pdsi  soil_moisture  water_deficit    vpd  n_signs_ok  best_abs
region                                                                                                                   
Klamath Mountains/California High North Coast Range  29 -0.348         -0.529          0.465  0.329           4     0.529
Northern Rockies                                     29 -0.130         -0.197          0.452  0.420           4     0.452
Idaho Batholith                                      29 -0.150          0.056          0.446  0.391           3     0.446
Canadian Rockies                                     29  0.158          0.165          0.437  0.377           2     0.437
Blue Mountains                                       29 -0.252 

**Finding.** The failure was **not uniform** — pooling hid a real regional signal.

Disaggregated to the 20 largest-burning JJA regions, the best-covariate |ρ| ranges from **0.086**
(Central Basin and Range — genuinely nothing) to **0.529** (Klamath Mountains). A single pooled fit
cannot represent a relationship that varies this much, and in two regions **inverts**.

**A coherent cluster of forest regions shows the expected physics.** Six regions clear |ρ| > 0.367,
the approximate two-sided p < 0.05 threshold at n = 29 (about 1 would be expected by chance among 20,
so six is not a multiple-comparisons artifact, though individual ranks within the six should not be
over-read):

| Region | pdsi | soil_moisture | water_deficit | vpd | signs correct |
| --- | ---: | ---: | ---: | ---: | ---: |
| Klamath Mountains / CA High North Coast Range | −0.348 | **−0.529** | **0.465** | 0.329 | 4/4 |
| Northern Rockies | −0.130 | −0.197 | **0.452** | 0.420 | 4/4 |
| Idaho Batholith | −0.150 | 0.056 | **0.446** | 0.391 | 3/4 |
| Canadian Rockies | 0.158 | 0.165 | **0.437** | 0.377 | 2/4 |
| Blue Mountains | −0.252 | −0.399 | **0.409** | 0.302 | 4/4 |
| North Cascades | −0.166 | −0.114 | **0.380** | 0.241 | 4/4 |

These are contiguous Pacific Northwest and Northern Rockies forest ecoregions — a geographically
coherent block, not a scatter. **11 of 20** regions have all four signs matching expectation.

**`pdsi` is the weakest of the four**, despite being the direct drought ask. Mean |ρ| across regions:
`water_deficit` 0.245, `vpd` 0.220, `soil_moisture` 0.183, `pdsi` **0.154**. The energy-balance
measures beat the drought index everywhere it matters.

**Two regions are fully inverted, and that is interpretable.** Mojave Basin and Range (0/4 signs;
pdsi **+0.255**, soil_moisture **+0.353**) and Snake River Plain (0/4) show *wetter* pre-seasons
associated with *more* burn — the grass-fire mechanism in arid systems, where antecedent moisture
grows the fine fuel that later cures and carries fire. These regions are **fuel-load-limited** rather
than fuel-dryness-limited. Mojave is also the most year-concentrated region in the panel (top year =
64% of its record), consistent with burn gated on an occasional wet-year fuel pulse. A pooled model
containing both mechanisms with opposite signs finds approximately nothing — which is what the
pooled rung found.

**Volatility and total acres are different rankings.** Interior Forested Lowlands is the largest
burner (14.5M acres) but has middling log-space volatility (log_sd 0.93) and the weakest covariate
response of any major region (best |ρ| = 0.195). Klamath ranks 7th by acres (4.0M) but is the **most
volatile** major region (log_sd 1.47, acres spanning 29 to 1,176,272) and has the strongest response.
Total acres is the wrong criterion for selecting a region to test a covariate against — the question
needs interannual variance to explain.

**What this establishes.** A descriptive correlation on the full record, not held-out skill: all 29
years, no forward-chaining split, no model. A ρ of 0.53 on 29 points is a wide confidence interval,
and correlation at this sample size does not demonstrate a stratified model would beat persistence
out of sample. What it does establish is that the pooled negative result is **an artifact of grain,
not an absence of signal**.

The covariate set currently has no way to express the fuel-load mechanism the Mojave/Snake River
inversion points to — LANDFIRE was rejected for carrying no interannual variance, but that inversion
is precisely a fuel-load signal.